# 06. 최종 리포트: 추천 알고리즘의 인과적 효과 분석

## Counterfactual Evaluation of Recommendation Algorithms

**데이터셋**: KuaiRec (완전관측) + KuaiRand-Pure (랜덤/추천 노출)  
**분석 기간**: Kuaishou 플랫폼 2022년 4월 22일 ~ 5월 8일  
**분석 범위**: EDA → A/B 테스트 → 인과추론 → Off-Policy Evaluation → 필터버블

---

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['font.family'] = 'Malgun Gothic'
plt.rcParams['axes.unicode_minus'] = False

FIG_DIR = Path('figures')
print('✅ 최종 리포트 환경 준비 완료')

## 1. Executive Summary

### 핵심 발견

| 분석 영역 | 핵심 결론 | 수치 근거 |
|-----------|----------|----------|
| **A/B 테스트** | 추천 알고리즘이 모든 참여 지표를 유의하게 향상 | CTR +153%, 좋아요 +269%, 댓글 +610% |
| **인과추론** | 추천 효과는 거의 모든 유저에게 양(+) | CATE > 0인 유저 99.3%, 평균 ATT = 0.263 |
| **OPE** | Direct Method 추정량이 가장 정확 | DM 오차 0.2%, IPS 92%, SNIPS 12% |
| **필터버블** | 추천이 콘텐츠 다양성을 체계적으로 감소 | 92.4% 유저의 Shannon Entropy 하락 |

### 비즈니스 시사점

> **"추천 알고리즘은 CTR을 153% 올리지만, 92.4%의 유저에서 콘텐츠 다양성을 감소시킨다."**  
> → 참여도와 다양성의 트레이드오프를 관리하는 하이브리드 전략이 필요하다.

---
## 2. 데이터 개요

### 2.1 KuaiRand-Pure — 랜덤 vs 추천 노출 실험

| 구분 | 랜덤 노출 (Control) | 추천 노출 (Treatment) |
|------|---------------------|----------------------|
| 노출 수 | 1,186,059 | 295,497 |
| 유저 수 | 27,285 | 25,877 |
| 영상 수 | 7,583 | 6,618 |
| 실험 설계 | Within-subject (동일 유저가 양쪽 모두 경험) |

- `is_rand=1`: 추천 피드에 랜덤 삽입된 영상 (자연 실험의 Control)
- `is_rand=0`: 알고리즘이 선택한 영상 (Treatment)

### 2.2 KuaiRec — 완전관측 행렬 (Ground Truth)

| 항목 | 값 |
|------|----|
| 유저 × 영상 | 1,411 × 3,327 |
| 관측 밀도 | 99.6% |
| 총 상호작용 | 4,676,570 |
| 용도 | Off-Policy Evaluation의 Ground Truth |

---
## 3. A/B 테스트 결과 (02_ab_test_basic)

In [ ]:
# A/B 테스트 결과 요약 테이블
ab_results = pd.DataFrame({
    '지표': ['CTR (is_click)', '좋아요 (is_like)', '팔로우 (is_follow)', 
             '댓글 (is_comment)', '공유 (is_forward)', '싫어요 (is_hate)', '장시청 (long_view)'],
    'Control': [0.1762, 0.0048, 0.0003, 0.0003, 0.0003, 0.0011, 0.0850],
    'Treatment': [0.4450, 0.0177, 0.0013, 0.0025, 0.0008, 0.0008, 0.3134],
    'Lift (%)': ['+152.6', '+269.1', '+410.1', '+610.0', '+145.6', '-31.7', '+268.9'],
    'p-value': ['< 1e-300'] * 7,
    'Bonferroni': ['✅ Sig'] * 7
})
print('=' * 80)
print('  A/B 테스트 결과: 추천(Treatment) vs 랜덤(Control) — z-test with Bonferroni')
print('=' * 80)
display(ab_results.style.hide(axis='index'))

In [ ]:
# A/B 시각화 — Forest Plot + Bootstrap CI
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

img1 = mpimg.imread(FIG_DIR / '02_forest_plot.png')
axes[0].imshow(img1)
axes[0].set_title('Forest Plot: 지표별 효과 크기', fontsize=13, fontweight='bold')
axes[0].axis('off')

img2 = mpimg.imread(FIG_DIR / '02_bootstrap_ci.png')
axes[1].imshow(img2)
axes[1].set_title('Bootstrap 95% CI: CTR 차이', fontsize=13, fontweight='bold')
axes[1].axis('off')

plt.tight_layout()
plt.savefig(FIG_DIR / '06_ab_summary.png', dpi=150, bbox_inches='tight')
plt.show()
print('\n📊 모든 7개 피드백 지표에서 통계적으로 유의한 차이 (Bonferroni 보정 후)')
print('📊 Cohen\'s d = 0.578 (중간 효과 크기) — 시청시간 기준')
print('📊 Statistical Power = 100% — MDE at 80% power = 0.006')

### 3.1 세그먼트별 분석

| 유저 그룹 | Control CTR | Treatment CTR | Lift |
|-----------|------------|--------------|------|
| Full Active | 0.169 | 0.417 | +146% |
| High Active | 0.190 | 0.488 | +156% |
| Middle Active | 0.198 | 0.526 | +166% |
| Low Active | 0.235 | 0.570 | +143% |

**발견**: Middle Active 유저에서 추천 효과(Lift)가 가장 크다 (+166%).  
Low Active 유저는 Base CTR이 높지만 Lift는 상대적으로 낮다 — 이미 관심사가 뚜렷하거나 소수 영상만 시청하는 패턴.

---
## 4. 인과추론 심화 (03_causal_inference)

In [ ]:
# PSM + CATE 시각화
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

for ax, (fname, title) in zip(axes.flat, [
    ('03_propensity_score.png', 'Propensity Score 분포'),
    ('03_love_plot.png', 'Love Plot: 매칭 전후 SMD'),
    ('03_cate_distribution.png', 'CATE 분포 (T-Learner)'),
    ('03_uplift_quadrant.png', 'Uplift 4분면 분류')
]):
    img = mpimg.imread(FIG_DIR / fname)
    ax.imshow(img)
    ax.set_title(title, fontsize=13, fontweight='bold')
    ax.axis('off')

plt.tight_layout()
plt.savefig(FIG_DIR / '06_causal_summary.png', dpi=150, bbox_inches='tight')
plt.show()

### 4.1 Propensity Score Matching 결과

- **매칭률**: 100% (caliper = 0.05, 10,164 pairs)
- **공변량 균형**: 매칭 후 모든 변수의 SMD < 0.1 (좋은 균형)
- **ATT(CTR)**: 0.263 — 나이브 차이(0.249) 대비 +5.2% 상향
  - 팔로우의 경우 나이브 추정 대비 62.6% 편향 감소 → 선택 편향이 가장 심한 지표

### 4.2 CATE (Conditional Average Treatment Effect)

- **평균 CATE**: 0.249 (추천 시 CTR이 약 25%p 상승)
- **99.3%** 의 유저가 양의 CATE → 추천이 거의 모든 유저에게 효과적
- **CATE 범위**: -0.39 ~ +0.77 (이질적 효과 존재)
- **핵심 변수**: 가입일수(34%), 팔로잉 수(20%), 팬 수(20%)

### 4.3 Uplift 4분면 분류

| 세그먼트 | 비율 | 의미 | 전략 |
|---------|------|------|------|
| **Persuadables** | 34.8% | 추천으로 행동 전환 가능 | 적극 추천 |
| **Sure Things** | 34.7% | 추천 없이도 클릭 | 다양성 확대 추천 |
| **Lost Causes** | 15.3% | 추천해도 반응 없음 | 콘텐츠 다각화 |
| **Sleeping Dogs** | 15.2% | 추천 시 오히려 이탈 위험 | 추천 빈도 줄이기 |

---
## 5. Off-Policy Evaluation (04_ope)

In [ ]:
# OPE 결과 시각화
fig, ax = plt.subplots(1, 1, figsize=(10, 6))
img = mpimg.imread(FIG_DIR / '04_ope_comparison.png')
ax.imshow(img)
ax.set_title('OPE 추정량 비교: DM vs IPS vs SNIPS', fontsize=14, fontweight='bold')
ax.axis('off')
plt.tight_layout()
plt.savefig(FIG_DIR / '06_ope_summary.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# OPE 정확도 비교
ope_results = pd.DataFrame({
    '정책': ['Popular', 'UserCF', 'Blend'],
    'Ground Truth': [0.662, 0.791, 0.834],
    'DM 추정': [0.662, 0.791, 0.832],
    'IPS 추정': [0.050, 0.067, 0.037],
    'SNIPS 추정': [0.750, 0.889, 0.917],
    'DM 오차(%)': ['0.00', '0.00', '0.24'],
    'IPS 오차(%)': ['92.46', '91.59', '95.51'],
    'SNIPS 오차(%)': ['13.29', '12.38', '9.91']
})
print('=' * 80)
print('  OPE 추정량 정확도 비교 (KuaiRec 완전관측 행렬 기준)')
print('=' * 80)
display(ope_results.style.hide(axis='index'))

print('\n💡 핵심 인사이트:')
print('   • DM(Direct Method): 거의 완벽한 추정 (오차 < 0.3%)')
print('   • IPS: 심각한 분산 문제 (오차 > 90%) — 높은 propensity 비율에 의한 분산 폭발')
print('   • SNIPS: IPS 대비 대폭 개선되었으나 여전히 10~13% 과대추정')
print('   • 완전관측 데이터가 있으면 DM이 최적, 없으면 SNIPS가 실용적 대안')

### 5.1 정책별 성능 비교 (Ground Truth)

| 정책 | Precision@10 | Recall@10 | 설명 |
|------|-------------|-----------|------|
| Random | 0.048 | 0.003 | 무작위 추천 (Baseline) |
| Popular | 0.671 | 0.115 | 인기 영상 추천 |
| UserCF | 0.794 | 0.139 | 유저 기반 협업 필터링 |
| **Blend** | **0.838** | **0.142** | Popular(30%) + UserCF(70%) 혼합 |

**Blend 정책**이 가장 높은 성능 — 인기도 신호와 개인화의 결합이 효과적.

---
## 6. 필터버블 분석 (05_filter_bubble)

In [ ]:
# 필터버블 시각화
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

for ax, (fname, title) in zip(axes.flat, [
    ('05_diversity_comparison.png', '다양성 지표 비교: Random vs Recommended'),
    ('05_user_entropy.png', '유저별 엔트로피 변화 분포'),
    ('05_long_tail.png', 'Long Tail: 인기 편중 분석'),
    ('05_daily_diversity.png', '일별 다양성 추이')
]):
    img = mpimg.imread(FIG_DIR / fname)
    ax.imshow(img)
    ax.set_title(title, fontsize=13, fontweight='bold')
    ax.axis('off')

plt.tight_layout()
plt.savefig(FIG_DIR / '06_filter_bubble_summary.png', dpi=150, bbox_inches='tight')
plt.show()

### 6.1 다양성 지표

| 지표 | Random | Recommended | 해석 |
|------|--------|------------|------|
| Shannon Entropy | 5.147 | 4.723 | 추천 시 다양성 8.2% 감소 |
| Gini Index | 0.758 | 0.809 | 추천 시 불평등 6.7% 증가 |
| 고유 영상 수 | 7,583 | 6,618 | 12.7% 적은 영상 노출 |

### 6.2 유저 수준 영향

- **92.4%** 의 유저가 추천 시 엔트로피 하락 (다양성 감소)
- 평균 엔트로피 차이: **-1.585** (Paired t-test, p < 1e-300)
- 7.6% 유저만 추천 시 다양성 증가 → 이들은 원래 특정 장르에 편향된 시청 패턴

### 6.3 Long Tail / 인기 편중

- **Random**: 상위 74.7% 영상이 80% 노출 차지 (비교적 균등)
- **Recommended**: 상위 **23.0%** 영상이 80% 노출 차지 (극단적 편중)
- 추천 알고리즘이 인기 콘텐츠에 트래픽을 집중시켜 니치 콘텐츠 발견 기회를 축소

---
## 7. 종합 결론 & 비즈니스 제안

In [ ]:
# 최종 트레이드오프 요약
tradeoff = pd.DataFrame({
    '차원': ['참여도 (CTR)', '참여도 (좋아요)', '참여도 (장시청)', 
             '다양성 (Entropy)', '다양성 (Long Tail)', '부정 반응 (싫어요)'],
    '추천 효과': ['+153%', '+269%', '+269%', '-8.2%', '-69% (커버리지)', '-32%'],
    '방향': ['🟢 긍정', '🟢 긍정', '🟢 긍정', '🔴 부정', '🔴 부정', '🟢 긍정'],
    '통계적 유의성': ['p < 1e-300', 'p < 1e-300', 'p < 1e-300', 
                    'p < 1e-300', 'N/A (기술통계)', 'p = 7.7e-08']
})
print('=' * 80)
print('  추천 알고리즘의 양면적 효과 — 참여도 vs 다양성 트레이드오프')
print('=' * 80)
display(tradeoff.style.hide(axis='index'))

### 7.1 비즈니스 제안

#### 제안 1: 유저 세그먼트별 차별화 전략

| 세그먼트 | 비율 | 추천 전략 |
|---------|------|----------|
| **Persuadables** (34.8%) | 추천 효과 최대 | 현행 알고리즘 유지, A/B 테스트로 최적화 지속 |
| **Sure Things** (34.7%) | 추천 없이도 참여 | 다양성 확대 추천 (Exploration 비율 ↑) |
| **Lost Causes** (15.3%) | 추천 효과 미미 | 콘텐츠 카테고리 다각화, 온보딩 개선 |
| **Sleeping Dogs** (15.2%) | 추천 시 이탈 위험 | 추천 빈도 ↓, 랜덤 탐색 기회 제공 |

#### 제안 2: 다양성-참여도 균형 메커니즘

1. **Exploration-Exploitation 비율 조정**: 현재 랜덤 삽입 비율(~20%)을 세그먼트별 차등 적용
2. **카테고리 쿼터제**: 추천 리스트의 최소 30%를 비주류 카테고리로 할당
3. **Long Tail 부스팅**: 인기 영상 외에 니치 콘텐츠를 주기적으로 노출

#### 제안 3: 추천 정책 배포 시 OPE 활용

- 새 알고리즘 배포 전 **DM 추정량**으로 사전 평가 (오차 < 0.3%)
- 완전관측 데이터 없을 경우 **SNIPS** 사용 (IPS 대비 7~8배 정확)
- 정기적으로 소규모 랜덤 노출 실험을 유지해 propensity 추정 품질 확보

### 7.2 분석 방법론 요약

```
┌─────────────────────────────────────────────────────────────────┐
│                    분석 파이프라인                               │
├─────────────────────────────────────────────────────────────────┤
│                                                                 │
│  01. EDA                                                        │
│   └── KuaiRand 12개 피드백 비교 + KuaiRec 완전관측 탐색          │
│                    ↓                                            │
│  02. A/B 테스트                                                  │
│   └── z-test, Welch's t, Mann-Whitney, Bootstrap CI              │
│   └── SRM 검정, Bonferroni 보정, 통계적 검정력 분석               │
│                    ↓                                            │
│  03. 인과추론                                                    │
│   └── PSM (Caliper NN), ATT 추정                                │
│   └── T-Learner CATE + Uplift 4분면                              │
│                    ↓                                            │
│  04. Off-Policy Evaluation                                      │
│   └── Random / Popular / UserCF / Blend 정책                    │
│   └── DM, IPS, SNIPS 추정량 vs Ground Truth 검증                 │
│                    ↓                                            │
│  05. 필터버블                                                    │
│   └── Shannon Entropy, Gini, Long Tail, 시계열 다양성            │
│                    ↓                                            │
│  06. 최종 리포트 (본 문서)                                       │
│   └── 종합 결론 + 비즈니스 제안                                  │
│                                                                 │
└─────────────────────────────────────────────────────────────────┘
```

### 7.3 한계점 및 향후 연구

| 한계 | 설명 | 개선 방향 |
|------|------|----------|
| Within-subject 설계 | 동일 유저의 랜덤/추천 노출이 독립이 아님 | Clustered SE, Mixed Effects Model |
| 단기 데이터 | 17일간 데이터로 장기 효과 추정 불가 | KuaiRand 전체(4개월) 분석 |
| 플랫폼 특수성 | Kuaishou 숏폼 → 타 플랫폼 일반화 한계 | 다중 플랫폼 비교 연구 |
| CATE 모델 | T-Learner 단일 모델 사용 | S-Learner, X-Learner, CausalForest 비교 |
| OPE 샘플 | 200명 샘플로 추정 | 전체 유저 + Doubly Robust 추가 |

---
## 8. 기술 스택 & 재현 가이드

```
Python 3.10+
├── 데이터: pandas, numpy
├── 통계: scipy, statsmodels
├── ML: scikit-learn, surprise
├── 인과추론: econml, causalml, scikit-uplift
├── 시각화: matplotlib, seaborn, plotly
└── 환경: Jupyter Notebook
```

**데이터 출처**:
- KuaiRec 2.0: https://zenodo.org/records/12594951
- KuaiRand-Pure: https://zenodo.org/records/12600781

**레퍼런스**:
- Gao et al. (2022) "KuaiRec: A Fully-observed Dataset and Insights for Evaluating Recommender Systems" (CIKM 2022)
- Gao et al. (2022) "KuaiRand: An Unbiased Sequential Recommendation Dataset with Randomly Exposed Videos" (CIKM 2022)

In [ ]:
print('=' * 80)
print('  📋 프로젝트 완료: 추천 알고리즘의 인과적 효과 분석')
print('=' * 80)
print()
print('  노트북 구성:')
print('    01_eda.ipynb              — 탐색적 데이터 분석')
print('    02_ab_test_basic.ipynb     — A/B 테스트 통계 검정')
print('    03_causal_inference.ipynb  — PSM + CATE + Uplift')
print('    04_ope.ipynb              — Off-Policy Evaluation')
print('    05_filter_bubble.ipynb     — 필터버블 다양성 분석')
print('    06_final_report.ipynb      — 종합 리포트 (본 문서)')
print()
print(f'  총 시각화: {len(list(Path("figures").glob("*.png")))}장')
print('  분석 완료 ✅')